In [ ]:
# Shared Gemini settings (run this cell before the model examples).
from pathlib import Path
import os
import sys

_gemini_root = next(
    (p for p in (Path.cwd() / "25125067_W05_03", Path.cwd(), *Path.cwd().parents)
     if (p / "gemini_setup.py").is_file()),
    None,
)
if _gemini_root is None:
    raise FileNotFoundError("Open this notebook from the 25125067_W05_03 folder.")
if str(_gemini_root) not in sys.path:
    sys.path.insert(0, str(_gemini_root))
from gemini_setup import load_environment

load_environment()


## Creating subagents

In [ ]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x ** 2

In [ ]:
from langchain.agents import create_agent

# create subagents

subagent_1 = create_agent(
    model= "google_genai:gemini-3.7-flash",
    tools=[square_root]
)

subagent_2 = create_agent(
    model= "google_genai:gemini-3.7-flash",
    tools=[square]
)

## Calling subagents

In [ ]:
from langchain.messages import HumanMessage

@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = subagent_1.invoke({"messages": [HumanMessage(content=f"Calculate the square root of {x}")]})
    return response["messages"][-1].text

@tool
def call_subagent_2(x: float) -> float:
    """Call subagent 2 in order to calculate the square of a number"""
    response = subagent_2.invoke({"messages": [HumanMessage(content=f"Calculate the square of {x}")]})
    return response["messages"][-1].text

## Creating the main agent

main_agent = create_agent(
    model='google_genai:gemini-3.7-flash',
    tools=[call_subagent_1, call_subagent_2],
    system_prompt="You are a helpful assistant who can call subagents to calculate the square root or square of a number.")

## Test

In [ ]:
question = "What is the square root of 456?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

In [ ]:
from pprint import pprint

pprint(response)